# CVD_평가 — 내부 합성 Test 성능 / 원본 Test 추론속도
이 노트북에서 학습된 GEVIS의 BOOLEAN·CHOICE·ORDINAL 성능과 속도를 확인합니다. 성능은 정답이 있는 합성 paired Test, 속도는 원본 Test 창의 TRAIN 참조 요청입니다. 두 입력 조건을 구분하며 실제 고장 정확도나 cycle/Edge 인수를 주장하지 않습니다.


In [ ]:
from pathlib import Path
import sys, importlib
PIPELINE_ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'src'/'pecvd').is_dir())
if str(PIPELINE_ROOT/'src') not in sys.path: sys.path.insert(0,str(PIPELINE_ROOT/'src'))
import pecvd.experiments.pipeline_settings as pipeline_settings
pipeline_settings=importlib.reload(pipeline_settings)
PIPELINE_SETTINGS=pipeline_settings.configure_pipeline(PIPELINE_ROOT)
if PIPELINE_SETTINGS.get('decision_system')!='fusion_answers':
    raise RuntimeError('STEP4 evaluates fusion_answers. Check config/experiments/gevis_pipeline.json.')
from pecvd.experiments.questions import PRIMARY_GOAL_VERSION
from pecvd.experiments.fusion_answers import QUESTIONS, ANSWER_CRITERIA
from pecvd.experiments.fusion_answers import SYSTEM as GEVIS_SYSTEM
print({'model':'fast GEVIS Fusion answers','decision_system':GEVIS_SYSTEM,
       'model_variant':PIPELINE_SETTINGS.get('model_variant'),'training_profile':PIPELINE_SETTINGS['training_profile']})
print({kind:QUESTIONS[kind] for kind in ('BOOLEAN','CHOICE','ORDINAL')})


In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src'/'pecvd').is_dir(): raise RuntimeError('Run from the project root or notebooks directory.')
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
if globals().get('PIPELINE_SETTINGS',{}).get('decision_system')!='fusion_answers':
    raise RuntimeError('Run shared-pipeline-settings with decision_system=fusion_answers first.')
SESSION_SIGNATURE='STEP4_GEVIS_FUSION_TEST_V1'
from pecvd.model_variants import active_run_tag
BACKBONE_RUN_TAG=active_run_tag()
SELECTED_STEPS=None  # None: all trained Steps; or provide a list.
TEST_WINDOWS_PER_STEP=32  # Held-out source windows used for controlled Test cases.
TEST_PAIRS_PER_STEP=2  # Original cross-run predictions only; not the Test window limit.
SEED=42
ERROR_DISPLAY_LIMIT=20
RUN_ORIGINAL_PREDICTIONS=False
if TEST_WINDOWS_PER_STEP<1 or not 1<=TEST_PAIRS_PER_STEP<=32 or ERROR_DISPLAY_LIMIT<1:
    raise ValueError('Positive Test/error limits and 1..32 original pairs required.')
print('Fast GEVIS: frozen trained weights; Test evaluation only; no CSV reads or fitting.')

SHOW_DETAILS=False
RUN_BOOTSTRAP=False
BENCHMARK_REPEATS=3
BENCHMARK_WARMUP=1


## 1. 학습된 Fast GEVIS 모델 불러오기

STEP 3에서 저장한 답변 헤드를 같은 STEP 1·2 가중치와 함께 불러옵니다.
모델·소스·설정이 맞지 않거나 학습된 헤드가 없으면 중단합니다. 임의 가중치로 평가하지 않습니다.


In [ ]:
import importlib
importlib.invalidate_caches()
import pecvd.experiments.routing_audit as runtime_refresh
runtime_refresh=importlib.reload(runtime_refresh)
runtime_refresh.refresh_evaluation_modules()
for _name in ('pecvd.experiments.fusion_metrics','pecvd.experiments.fusion_answers','pecvd.experiments.gevis_system'):
    importlib.reload(importlib.import_module(_name))
from pecvd.experiments.evaluation import load_experiment_runtime
from pecvd.experiments.fusion_answers import require_head
REFERENCE_RUNTIME=load_experiment_runtime(ROOT)
if getattr(REFERENCE_RUNTIME.module,'DECISION_SYSTEM',None)!='fusion_answers':
    raise RuntimeError('Loaded runtime is not fusion_answers. Restart the kernel and check the active configuration.')
FUSION_HEAD=require_head(REFERENCE_RUNTIME.module)
MODEL_INFO=pd.DataFrame([dict(model='fast GEVIS: question Fusion + selected-center answer head',
    answer_head_parameters=sum(p.numel() for p in FUSION_HEAD.parameters()),
    loaded_steps=len(REFERENCE_RUNTIME.encoders),evaluation_split='test',weights_updated=False,
    answer_rule='learned logits argmax',fault_probability_calibrated=False)])
display(MODEL_INFO)


## 2. Fast model — Test: 세 답변 성능

분리된 Test 회차의 원본 창에 통제 변화를 추가하고 주입 전 창을 참조로 제공합니다. 이 paired 합성 진단은 새 입력의 TRAIN 참조 정확도와 다릅니다. 이전 Test 결과를 개발에 참고한 이력이 있어 최종 미열람 holdout으로 부르지 않습니다.

기본 표는 답변별 F1·정확도·변경 사례 정확도·강도 오차입니다. 상세 혼동표와 오답은 SHOW_DETAILS=True, run 단위 bootstrap 신뢰구간은 RUN_BOOTSTRAP=True로 선택합니다. 같은 run의 질문·창은 독립 표본이 아닙니다. ORDINAL은 합성 진폭/지연 단계이며 실제 고장 심각도가 아닙니다.


In [ ]:
import importlib
from dataclasses import replace
from time import perf_counter
from pecvd.experiments.gevis_system import run_gevis_experiment
from pecvd.experiments.fusion_answers import FusionConfig
import pecvd.experiments.fusion_metrics as fusion_metrics
fusion_metrics=importlib.reload(fusion_metrics)
TEST_CONFIG=replace(FusionConfig(),seed=SEED,max_windows_per_step=TEST_WINDOWS_PER_STEP)
_test_started=perf_counter()
FUSION_TEST=run_gevis_experiment(ROOT,runtime=REFERENCE_RUNTIME,split='test',steps=SELECTED_STEPS,config=TEST_CONFIG)
TEST_EVALUATION_SECONDS=perf_counter()-_test_started
from pecvd.experiments import fast_assessment
fast_assessment.compact(FUSION_TEST,display,evidence_scope='PECVD_paired_synthetic_Test',show_details=globals().get('SHOW_DETAILS',False))
REPORT=FUSION_TEST;SUMMARY=REPORT.summary;DETAILS=REPORT.details;BY_STEP=REPORT.by_step;ERROR_CASES=REPORT.errors
print(f'Test evaluation elapsed: {TEST_EVALUATION_SECONDS:.2f}s (includes case preparation; not pure inference latency).')
print('Actual learned answer metrics on held-out paired perturbations; not verified equipment-fault performance.')


## 3. 원본 Test 창의 전체 요청 속도

워밍업 뒤 같은 TRAIN 참조 추론 경로에서 전체 센서의 세 답변을 측정합니다. 정확도 표의 합성 입력과는 다른 시험입니다. 평균50ms 목표와 비교하되 파일 읽기·수집·전송·대기열은 제외합니다.


In [ ]:
from pecvd.experiments import inference_benchmark as inference_timing
TIMING_CASES=inference_timing.pecvd_cases(REFERENCE_RUNTIME,steps=SELECTED_STEPS,
    config=replace(TEST_CONFIG,max_windows_per_step=2))
TIMING_REPORT=inference_timing.benchmark(REFERENCE_RUNTIME,TIMING_CASES,
    repeats=BENCHMARK_REPEATS,warmup=BENCHMARK_WARMUP,domain='PECVD',progress=print)
TIMING_ASSESSMENT=fast_assessment.compact(TIMING_REPORT,display,evidence_scope='PECVD_original_TRAIN_reference_requests')


## 4. 선택 센서 문맥 검증 — 선택 출력

같은 변경 파형에서 선택 센서를 바꾼 쌍을 모두 맞혔는지(`joint_answer_accuracy`),
다른 센서만 바뀌었는데 선택 센서도 바뀌었다고 잘못 답하는지(`false_change_rate`) 확인합니다.
이 지표만으로 질문·Cross-attention의 독립적인 기여가 입증되지는 않습니다.


In [ ]:
if SHOW_DETAILS:
    print('Selected-center swap: both answers correct')
    display(REPORT.diagnostics['center_swap'])
    print('Off-center changes: false change answers for the selected center')
    display(REPORT.diagnostics['off_center'])
    
    print('Changed selected sensor only: accuracy by direction, span recipe and type')
    display(REPORT.diagnostics['intervention_metrics'])
    
    print('Delay: native sample lags; effective changed centers only')
    display(REPORT.diagnostics['delay_metrics'])
    print('Reconstruction MSE evidence, not a fault probability or change-type label')
    display(REPORT.diagnostics['reconstruction_metrics'])
    
if RUN_BOOTSTRAP:
    from pecvd.experiments import evaluation_audit
    display(evaluation_audit.run_bootstrap(REPORT.details))


## 5. 혼동표와 실제 모델 오답 — 선택 출력

혼동표의 행은 정답, 열은 모델 예측입니다. 아래 오답은 알려진 합성 변화의 정답과 모델의
예측을 나란히 보여줍니다. 전체 오답은 `ERROR_CASES`, 전체 예측은 `DETAILS`에 있습니다.
화면에는 설정한 개수만 표시하며 결과를 자동 저장·전송하지 않습니다.


In [ ]:
if SHOW_DETAILS:
    display(REPORT.diagnostics['confusion'])
    if ERROR_CASES.empty:
        print('No incorrect answers in the evaluated Test cases.')
    else:
        ERROR_PREVIEW=ERROR_CASES.head(ERROR_DISPLAY_LIMIT).copy()
        ERROR_PREVIEW['expected_answer']=[ANSWER_CRITERIA[k][int(y)] for k,y in zip(ERROR_PREVIEW.answer_type,ERROR_PREVIEW.label)]
        ERROR_PREVIEW['predicted_answer']=[ANSWER_CRITERIA[k][int(y)] for k,y in zip(ERROR_PREVIEW.answer_type,ERROR_PREVIEW.prediction)]
        _error_columns=['step','answer_type','center_sensor','expected_answer','predicted_answer','injected','center_is_perturbed']
        display(ERROR_PREVIEW[[c for c in _error_columns if c in ERROR_PREVIEW]])
        print(f'Showing {len(ERROR_PREVIEW)} of {len(ERROR_CASES)} incorrect answers.')


## 6. 원본 공정 비교 예측 — 독립 정답 없음

이 셀은 원본 Test 회차와 참조 회차를 비교한 답변 예시입니다. 선택 센서를 비교 쌍마다 순환합니다.
독립적인 실제 이상 정답이 없으므로 이 결과에 정확도·recall·F1을 붙이지 않습니다.
위 Test 성능 표와 다른 결과이며 `RUN_ORIGINAL_PREDICTIONS=False`로 생략할 수 있습니다.


In [ ]:
if RUN_ORIGINAL_PREDICTIONS:
    from pecvd.experiments.gevis_system import run_gevis_predictions
    FUSION_ORIGINAL=run_gevis_predictions(ROOT,runtime=REFERENCE_RUNTIME,steps=SELECTED_STEPS,
        pairs_per_step=TEST_PAIRS_PER_STEP,seed=SEED)
    display(FUSION_ORIGINAL.summary)
    _original_columns=['step','pair_index','center_sensor','answer_type','answer','accepted']
    display(FUSION_ORIGINAL.details[[c for c in _original_columns if c in FUSION_ORIGINAL.details]].head(ERROR_DISPLAY_LIMIT))
    print('Original cross-run answers: accuracy unavailable without independent truth.')


## 7. Recorded split audit (optional)

Checks preprocessing partitions and recorded head-fit/selection runs. Disjoint partitions do not certify an untouched final holdout. Set RUN_SPLIT_AUDIT=True to read the recorded cache identities.


In [ ]:
RUN_SPLIT_AUDIT=False
if RUN_SPLIT_AUDIT:
    from pecvd.experiments import evaluation_audit
    display(evaluation_audit.split_audit(REFERENCE_RUNTIME))
